In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.appName("UPI-transactions").getOrCreate()

In [0]:

username = dbutils.secrets.get(scope="credentials", key="RED_PANDA_USERNAME")
password = dbutils.secrets.get(scope="credentials", key="RED_PANDA_PASSWORD")
transactions_raw_binary = (spark.readStream
                   .format("kafka")
                   .option("kafka.bootstrap.servers", "d9s20tguj23020u21opg.any.ap-south-1.mpx.prd.cloud.redpanda.com:9092")
                   .option("subscribe", "UPI_Transactions")
                   .option("startingOffsets", "earliest")
                   .option("kafka.security.protocol", "SASL_SSL")
                   .option("kafka.sasl.jaas.config", f"kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username='{username}' password='{password}';")
                   .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
                   .load()
                  )

In [0]:
transactions_string_convert = (transactions_raw_binary
                                .select(col("key").cast("string"),
                                        col("value").cast("string"),
                                        col("topic"),
                                        col("partition"),
                                        col("offset"),
                                        col("timestamp"),
                                        col("timestampType")
                                        )
                                )

In [0]:
query = (transactions_string_convert.writeStream
                   .format("delta")
                   .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/Bronze/")
                   .outputMode("append")
                   .trigger(availableNow = True)
                   .queryName("transactions_bronze")
                   .toTable("upi_transactions_bronze")
        )

> # **Unloading Settlement File from S3**

In [0]:
settlement_schema = StructType([
    StructField("txn_id", StringType()),
    StructField("rrn", StringType()),
    StructField("settlement_amount", DecimalType(10,2)),
    StructField("settlement_status", StringType()),
    StructField("settlement_date", DateType()),
    StructField("settled_timestamp", TimestampType())
])

In [0]:
settled_read = (
                spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format","csv")
                .option("header","true")
                .schema(settlement_schema)
                .option("cloudFiles.schemaEvolutionMode","rescue")
                .load("s3://settlement-file-databricks/")
                )

write_query = (
                settled_read.writeStream
                .format("delta")
                .option("checkpointLocation","/Volumes/workspace/default/checkpoints/settlement_s3_unload/")
                .outputMode("append")
                .trigger(availableNow=True)
                .queryName("gold_s3_unload")
                .toTable("workspace.default.settlement_data")
            )